# Exact Gaussian Solutions

This notebook demonstrates the analytical (exact) solutions for OTP-FM with Gaussian marginals.

Under the Gaussian ansatz, the multi-marginal optimal transport problem with intermediate marginal constraints admits closed-form solutions. This provides ground truth for validating the neural network approximations.

In [ ]:
import torch
import numpy as np

# Import the exact Gaussian solver
from experiments.gaussian.solver import GaussianMarginalSolver

# Import built-in plotting utilities
from experiments.gaussian.plotting import plot_trajectories_middle_marginal_1d

## 1. Define Gaussian Marginals

We define three Gaussian distributions with different means and covariances.

In [ ]:
# 1D Example: Three Gaussians
dim = 1

# Source: N(0, 1)
mu0 = np.array([0.0])
cov0 = np.array([[1.0]])

# Intermediate at t=0.5: N(3, 0.25)
mu_half = np.array([3.0])
cov_half = np.array([[0.25]])

# Target: N(1, 2)
mu1 = np.array([1.0])
cov1 = np.array([[2.0]])

print("Marginals:")
print(f"  t=0.0: N({mu0[0]}, {cov0[0,0]})")
print(f"  t=0.5: N({mu_half[0]}, {cov_half[0,0]})")
print(f"  t=1.0: N({mu1[0]}, {cov1[0,0]})")

## 2. Create Solver and Compute Exact Solution

In [ ]:
# Create solver with intermediate marginal
solver = GaussianMarginalSolver(
    d=dim,
    source_mean=mu0,
    source_cov=cov0,
    target_mean=mu1,
    target_cov=cov1,
    tks=[0.5],
    marginal_means=[mu_half],
    marginal_covs=[cov_half],
)

# Solve using shooting method
success = solver.solve()
print(f"Solver converged: {success}")

## 3. Sample Exact Trajectories

In [ ]:
# Sample initial conditions from source
n_samples = 100
x0 = np.random.randn(n_samples, dim) * np.sqrt(cov0[0, 0]) + mu0

# Generate exact trajectories
t_eval = np.linspace(0, 1, 50)
trajectories = solver.sample_trajectories(x0, t_eval)

print(f"Trajectory shape: {trajectories.shape}")

## 4. Visualize Exact Solution

In [ ]:
means = [mu0[0], mu_half[0], mu1[0]]
stds = [np.sqrt(cov0[0, 0]), np.sqrt(cov_half[0, 0]), np.sqrt(cov1[0, 0])]

# Normalized initial points (relative to source)
x0s_normalized = (x0[:, 0] - mu0[0]) / np.sqrt(cov0[0, 0])

plot_trajectories_middle_marginal_1d(
    means=means,
    stds=stds,
    x0s=x0s_normalized,
    t_k=np.array([0.5]),
    xs=trajectories[:, :, 0].T,  # (n_samples, n_timesteps)
    t_eval=t_eval,
    lambda_width=0.1,
    lambda_type="gaussian",
    title="Exact Gaussian Trajectories",
    n_trajectories=50,
    show=True,
)

## 5. Compare Learned vs Exact

In [ ]:
from collections import OrderedDict
from otpfm import OTPFM, Curriculum
from otpfm.potentials import W2InfPotential

# Create and train a neural network model
potentials = OrderedDict({
    0.5: W2InfPotential(tk=0.5, strength=100.0, lambda_type='gaussian', width=0.1)
})

model = OTPFM(
    d=dim,
    tks=[0.5],
    potentials=potentials,
    flownet_args={'hidden_dim': 64, 'num_hidden_layers': 2}
)

# Quick training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 200

# Create curriculum for OTP alpha scheduling
curriculum = Curriculum(total_iterations=n_epochs, schedule="sigmoid")

for epoch in range(n_epochs):
    model.train()
    
    # Sample from marginals
    x0_batch = torch.randn(64, dim) * np.sqrt(cov0[0, 0]) + mu0[0]
    x_half_batch = torch.randn(64, dim) * np.sqrt(cov_half[0, 0]) + mu_half[0]
    x1_batch = torch.randn(64, dim) * np.sqrt(cov1[0, 0]) + mu1[0]
    
    xs = torch.stack([x0_batch, x_half_batch, x1_batch], dim=1)
    otp_alpha = curriculum(epoch)
    
    loss = model.forward_with_loss(xs, otp_alpha)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.update_ema()

print("Training complete")

In [ ]:
# Sample from learned model
model.eval()
x0_test = torch.randn(n_samples, dim) * np.sqrt(cov0[0, 0]) + mu0[0]

with torch.no_grad():
    learned_traj, t_learned = model.sample(x0_test, n_steps=50, ema=True)

learned_traj = learned_traj.numpy()
t_learned = t_learned.numpy()

# Compute exact trajectories with same initial conditions
exact_traj = solver.sample_trajectories(x0_test.numpy(), t_learned)

In [ ]:
# Plot comparison using built-in plotting
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normalized initial points for visualization
x0s_normalized = (x0_test[:, 0].numpy() - mu0[0]) / np.sqrt(cov0[0, 0])

# Exact solution
plot_trajectories_middle_marginal_1d(
    means=means,
    stds=stds,
    x0s=x0s_normalized,
    t_k=np.array([0.5]),
    xs=exact_traj[:, :, 0].T,
    t_eval=t_learned,
    lambda_width=0.1,
    lambda_type="gaussian",
    title="Exact Solution",
    n_trajectories=50,
    fig=fig,
    ax=axes[0],
    show=False,
    close=False,
)

# Learned solution
plot_trajectories_middle_marginal_1d(
    means=means,
    stds=stds,
    x0s=x0s_normalized,
    t_k=np.array([0.5]),
    xs=learned_traj[:, :, 0].T,
    t_eval=t_learned,
    lambda_width=0.1,
    lambda_type="gaussian",
    title="Learned Solution (OTP-FM)",
    n_trajectories=50,
    fig=fig,
    ax=axes[1],
    show=False,
    close=False,
)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. Computing exact solutions for multi-marginal OT with Gaussian marginals
2. Training OTP-FM to approximate these solutions
3. Quantitative comparison between exact and learned trajectories

The exact solver provides ground truth for validating the neural network approximations and understanding the theoretical properties of the optimal transport problem with intermediate marginal constraints.